In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
root_dir = 'manipulation/metrics'
os.chdir(root_dir)

In [3]:
merged_ab = pd.read_csv('mt_exp_dataset/merged_ab.csv')
merged_ab

,mt,id,question_id,file_id,model_name,exp_score,time,nll,Nback,word_speed,line_speed,pixel_speed
0,1.0,54,85,54,alpaca-13b,1.25,24957,172.899567,1.5,324.116883,4159.500000,2.092361
1,2.0,54,85,54,llama-13b,1.75,30384,232.750596,2.0,297.882353,3798.000000,1.827504
2,1.0,54,85,54,alpaca-13b,1.25,50345,172.899567,1.5,653.831169,8390.833333,4.220857
3,2.0,54,85,54,llama-13b,1.75,40090,232.750596,0.0,393.039216,5011.250000,2.411290
4,1.0,54,85,54,alpaca-13b,2.00,19601,172.899567,3.5,254.558442,3266.833333,1.178940
...,...,...,...,...,...,...,...,...,...,...,...,...
1135,2.0,183,92,183,vicuna-13b-v1.2,1.00,14933,214.867607,1.0,219.602941,2133.285714,1.056737
1136,1.0,183,92,183,gpt-3.5-turbo,2.00,17582,216.941709,1.5,217.061728,2197.750000,1.244194
1137,2.0,183,92,183,vicuna-13b-v1.2,1.00,16483,214.867607,1.5,242.397059,2354.714286,0.945212
1138,1.0,183,92,183,gpt-3.5-turbo,1.25,34196,216.941709,1.0,422.172840,4274.500000,2.419888


### Determine random factors

In [4]:
print(len(merged_ab['id'].unique()))
# 51 unique ids

print(len(merged_ab['question_id'].unique()))
# 4 unique question_ids

print(len(merged_ab['file_id'].unique()))
# 51

51
4
51


In [6]:
import rpy2

%load_ext rpy2.ipython
%R require("ggplot2")
%R require("data.table")

Loading required package: ggplot2
Need help? Try Stackoverflow: https://stackoverflow.com/tags/ggplot2


Loading required package: data.table
data.table 1.14.8 using 1 threads (see ?getDTthreads).  Latest news: r-datatable.com
**********
This installation of data.table has not detected OpenMP support. It should still work but in single-threaded mode.
This is a Mac. Please read https://mac.r-project.org/openmp/. Please engage with Apple and ask them for support. Check r-datatable.com for updates, and our Mac instructions here: https://github.com/Rdatatable/data.table/wiki/Installation. After several years of many reports of installation problems on Mac, it's time to gingerly point out that there have been no similar problems on Windows or Linux.
**********


### Models without random factors

In [20]:
%%R -i merged_ab
dt <- data.table(merged_ab)

m0 <- lm(exp_score ~ nll, data=dt)
summary(m0)


Call:
lm(formula = exp_score ~ nll, data = dt)

Residuals:
     Min       1Q   Median       3Q      Max 
-1.50176 -0.25008 -0.00008  0.25017  1.50045 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept)  1.504e+00  2.976e-02  50.522   <2e-16 ***
nll         -1.677e-05  1.257e-04  -0.133    0.894    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.3364 on 1138 degrees of freedom
Multiple R-squared:  1.563e-05,	Adjusted R-squared:  -0.0008631 
F-statistic: 0.01778 on 1 and 1138 DF,  p-value: 0.8939



In [21]:
%%R -i merged_ab
dt <- data.table(merged_ab)

m <- lm(exp_score ~ model_name, data=dt)
summary(m)


Call:
lm(formula = exp_score ~ model_name, data = dt)

Residuals:
     Min       1Q   Median       3Q      Max 
-1.50397 -0.25397 -0.00397  0.24823  1.51685 

Coefficients:
                          Estimate Std. Error t value Pr(>|t|)    
(Intercept)                1.48315    0.02060  72.000   <2e-16 ***
model_nameclaude-v1        0.02568    0.04192   0.613    0.540    
model_namegpt-3.5-turbo    0.01205    0.02933   0.411    0.681    
model_namegpt-4            0.08442    0.05905   1.430    0.153    
model_namellama-13b        0.02082    0.02956   0.704    0.481    
model_namevicuna-13b-v1.2  0.02313    0.02997   0.772    0.440    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.3366 on 1134 degrees of freedom
Multiple R-squared:  0.002103,	Adjusted R-squared:  -0.002297 
F-statistic: 0.478 on 5 and 1134 DF,  p-value: 0.7929



In [22]:
%%R -i merged_ab
dt <- data.table(merged_ab)

m <- lm(exp_score ~ nll + log(time) + Nback + word_speed + line_speed + pixel_speed, data=dt)
summary(m)


Call:
lm(formula = exp_score ~ nll + log(time) + Nback + word_speed + 
    line_speed + pixel_speed, data = dt)

Residuals:
     Min       1Q   Median       3Q      Max 
-1.53153 -0.25138  0.00515  0.24665  1.50624 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept)  2.554e+00  3.065e-01   8.332 2.27e-16 ***
nll          3.482e-04  1.379e-04   2.525 0.011702 *  
log(time)   -1.283e-01  3.466e-02  -3.701 0.000225 ***
Nback       -6.400e-04  2.981e-03  -0.215 0.830017    
word_speed   9.661e-04  2.525e-04   3.827 0.000137 ***
line_speed  -2.923e-05  2.025e-05  -1.443 0.149232    
pixel_speed -3.779e-02  1.148e-02  -3.293 0.001022 ** 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.3305 on 1133 degrees of freedom
Multiple R-squared:  0.0386,	Adjusted R-squared:  0.03351 
F-statistic: 7.582 on 6 and 1133 DF,  p-value: 5.419e-08



In [7]:
%%R -i merged_ab
dt <- data.table(merged_ab)

m <- lm(mt ~ nll + log(time) + Nback + word_speed + line_speed + pixel_speed, data=dt)
summary(m)


Call:
lm(formula = mt ~ nll + log(time) + Nback + word_speed + line_speed + 
    pixel_speed, data = dt)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.81944 -0.44038  0.00115  0.42236  1.22119 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept)  3.307e+00  4.019e-01   8.228 5.19e-16 ***
nll         -1.421e-03  1.808e-04  -7.860 8.91e-15 ***
log(time)   -1.621e-01  4.544e-02  -3.568 0.000375 ***
Nback       -3.904e-03  3.908e-03  -0.999 0.317970    
word_speed   6.004e-04  3.310e-04   1.814 0.069965 .  
line_speed   8.839e-06  2.655e-05   0.333 0.739303    
pixel_speed -5.476e-02  1.505e-02  -3.640 0.000285 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.4333 on 1133 degrees of freedom
Multiple R-squared:  0.1421,	Adjusted R-squared:  0.1375 
F-statistic: 31.27 on 6 and 1133 DF,  p-value: < 2.2e-16

